# Lesson 8 | How do we know the hardware is correct?

Last lesson produced a readable **Register-Transfer Level (RTL)** neuron. “Looks plausible” is not verification. Today asks:
> **Without a physical FPGA board, how can we drive RTL inputs, observe outputs, and make errors fail automatically?**

Primary concept: **hardware simulation and the testbench**.


## 1. Concept ledger

**Known:** module/port, combinational path, and clocked register update.

**New:** simulation, testbench, waveform, and Device Under Test (DUT).

**Preview:** synthesis and physical FPGA execution are taught later as separate concepts.


## 2. Simulation, testbench, and waveform

**Simulation** executes HDL event and assignment rules in software. A **testbench** instantiates the circuit, generates clock, reset, and inputs, then compares outputs with expected values. The instantiated circuit is the **Device Under Test (DUT)**.

A **waveform** displays signal changes against simulation time. Inspect state updates after positive edges and clearing after reset edges. Automatic comparisons check expected results; waveforms show when changes happen.


## 3. Keep the testbench separate from design RTL

A testbench may use delays, logging, `$fatal`, and `$finish`; these are not logic to synthesize into the FPGA.

The `` `timescale 1ns/1ps `` line sets a 1 ns time unit and 1 ps simulation precision. `#4`, `#1`, and `#5` advance testbench time.


## 4. Write the oracle before viewing a waveform

First derive expected values from the known contract:


In [ ]:
state = 0
threshold = 4
print('cycle | input | before | candidate | spike | after')
for cycle, current in enumerate([1, 1, 1, 1, 2, 2]):
    before = state
    candidate = before + current
    spike = candidate >= threshold
    state = 0 if spike else candidate
    print(f'{cycle:5d} | {current:5d} | {before:6d} | {candidate:9d} | {int(spike):5d} | {state:5d}')


## 5. Port connection syntax

`tutorial_if_neuron dut (...)` creates an instance named `dut`. In `.clk(clk)`, the name after the dot is the module port; the expression in parentheses is the testbench signal. `.clk(test_clock)` would connect a differently named signal.

```systemverilog
tutorial_if_neuron dut (
    .clk(clk), .rst_n(rst_n),
    .input_current(input_current), .threshold(threshold),
    .reset_value(reset_value), .membrane_v(membrane_v), .spike(spike)
);
```

Check each input connection and observed output.


## 6. How automatic checks execute

This is the key task from the repository testbench. `task` / `endtask` declare a callable procedure; `automatic` gives each call independent local storage.

```systemverilog
task automatic apply_and_check(
    input logic signed [7:0] current,
    input logic signed [7:0] expected_v,
    input logic expected_spike
);
    begin
        input_current = current;
        #4 clk = 1'b1;
        #1;
        if (membrane_v !== expected_v)
            $fatal(1, "membrane_v got=%0d expected=%0d", membrane_v, expected_v);
        if (spike !== expected_spike)
            $fatal(1, "spike got=%0d expected=%0d", spike, expected_spike);
        #5 clk = 1'b0;
    end
endtask
```

Clock is low before a call. Drive input, wait 4 ns, generate a positive edge, wait another 1 ns for nonblocking updates, then check outputs. After 5 ns lower the clock. Reading immediately in the same execution region as the edge may observe old state. This `#1` schedules testbench sampling; physical delays require implementation timing analysis.

`!==` compares four-state values, exposing unexpected X/Z. `$fatal` reports an error and terminates simulation; `%0d` prints decimal values. An `initial` block starts once, applies reset, and calls the task. `$display` prints PASS and `$finish` ends normally. A separate `initial` block fails after `#1000` if simulation has not finished.


## 7. What do the command-line steps do?

```bash
mkdir -p build/lesson08
iverilog -g2012 -o build/lesson08/tutorial_if_neuron.vvp \
  rtl/learning/tutorial_if_neuron.sv \
  tb/learning/tutorial_if_neuron_tb.sv
cd build/lesson08 && vvp tutorial_if_neuron.vvp
```

The first tool compiles SystemVerilog into a simulation file; the second actually runs it.


## 8. Run: actual simulation when a simulator is installed

The Notebook keeps simulation artifacts under `build/lesson08/` instead of deleting the waveform with a temporary directory.


In [ ]:
from pathlib import Path
import shutil, subprocess

def repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'pyproject.toml').exists() and (p/'lessons').exists():
            return p
    raise FileNotFoundError('Run inside FPGA-FlyBrain')

root = repo_root()
iverilog = shutil.which('iverilog')
vvp = shutil.which('vvp')
if not (iverilog and vvp):
    print('Icarus Verilog not found; RTL simulation did not run.')
else:
    build = root/'build'/'lesson08'
    build.mkdir(parents=True, exist_ok=True)
    sim = build/'tutorial_if_neuron.vvp'
    subprocess.run([
        iverilog, '-g2012', '-o', str(sim),
        str(root/'rtl/learning/tutorial_if_neuron.sv'),
        str(root/'tb/learning/tutorial_if_neuron_tb.sv')
    ], check=True, cwd=build)
    result = subprocess.run([vvp, str(sim)], check=True, text=True, capture_output=True, cwd=build)
    print(result.stdout.strip())
    print('Waveform saved at:', build/'tutorial_if_neuron.vcd')


## 9. Waveform / VCD

The testbench uses `$dumpfile/$dumpvars` to create a **Value Change Dump (VCD)** file at `build/lesson08/tutorial_if_neuron.vcd`.

If GTKWave or Surfer is available, open it there. The self-checking pass/fail does not depend on having a waveform viewer.


## 10. Which layer do you inspect first on failure?

1. Is the contract/oracle correct?
2. Does the testbench drive/sample at the intended time?
3. Is the RTL combinational/register update wrong?

Do not randomly edit RTL until the test turns green.


## 11. Try It: deliberately create a detectable bug

On a temporary copy, change `>=` to `>`. Predict which boundary vector fails, run the testbench, then restore the original code.


## Exercise

[Lesson 8 exercise: write a minimal oracle for a self-checking testbench](../../exercises/en/08_testbench_waveform_simulation.ipynb)

The standalone workbook checks this lesson's semantic understanding; the real RTL/testbench experiment in the lesson remains part of the course.

## 12. AI Task

Give AI one concrete failure message and ask for evidence-based inspection points across oracle, testbench timing, and RTL before it suggests any change.


## 13. Human Check

Without AI, explain simulation vs physical FPGA, why a testbench is not design RTL, the complementary roles of self-checking tests and waveforms, and why a Python/RTL mismatch does not automatically make Python correct.


## 14. Engineering Handoff

This establishes RMD-005 / RMD-005A verification habits. Formal LIF RTL will use Python fixed-point vectors as its direct oracle; the tutorial testbench does not replace formal verification.


## 15. Project Trace

- Lesson: `LSN-008`
- Mapping: `RMD-005 / RMD-005A`
- Prepared verification level: `L3 RTL unit simulation`


## 16. Exit Ticket

You can distinguish compile from simulation, design RTL from testbench, self-checking assertions from waveform inspection, and avoid confusing “simulation passed” with “runs on FPGA.”
